# PRUEBA


In [14]:
import pandas as pd 
import optuna
from catboost import CatBoostClassifier
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from preprocessing.preprocessor import looe
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score
import numpy as np
import matplotlib.pyplot as plt

In [4]:
# We read the partitions
X_train = pd.read_parquet("../data/cleaned/X_train.parquet")
X_test = pd.read_parquet("../data/cleaned/X_test.parquet")
y_train = pd.read_parquet("../data/cleaned/y_train.parquet").squeeze()
y_test = pd.read_parquet("../data/cleaned/y_test.parquet").squeeze()

# Check that all has been loaded correctly
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)


(10045, 32)
(10045,)
(4948, 32)
(4948,)


In [ ]:


# PASO 1: Lees el cuaderno de notas (Cargar el estudio que ya tenías guardado)

cat_study = optuna.load_study(
    study_name="cat_study_native",
    storage="sqlite:///../hyperparameter_tuning/optuna_studies/cat_study.db"
)
# PASO 2: Le preguntas cuál fue la mejor combinación que encontró
mejores_parametros = cat_study.best_params

# (Opcional pero muy recomendado) Imprimes los parámetros para ver qué eligió la máquina
print("Los mejores parámetros encontrados fueron:")
print(mejores_parametros)

# PASO 3: Creas el modelo, inyectas la receta y lo entrenas con tus datos reales
# El doble asterisco (**) "desempaqueta" el diccionario para meter los parámetros uno a uno
modelo_catboost_final = CatBoostClassifier(**mejores_parametros, random_seed=42, verbose=False, thread_count=-1, cat_features = looe)



# 1. Sacamos las probabilidades honestas usando Cross-Validation
# Importante: Le pedimos 'predict_proba' en lugar de las predicciones normales
print("Calculando probabilidades out-of-fold (esto puede tardar un poco)...")
probabilidades_cv = cross_val_predict(
    modelo_catboost_final, 
    X_train, 
    y_train, 
    cv=5, 
    method='predict_proba'
)

# 2. Aislamos solo las probabilidades de que sea Clase 0 (la primera columna)
probs_clase_0_cv = probabilidades_cv[:, 0]

# 3. Creamos nuestra "Realidad Binaria" basada en y_train
y_train_binario = (y_train == 0).astype(int)

# 4. Probamos todos los umbrales
umbrales = np.arange(0.01, 1.0, 0.01)
f1_scores_cv = []

for umbral in umbrales:
    # Si la probabilidad supera el umbral, asignamos 1 (Clase 0), si no, 0
    predicciones_temporales = (probs_clase_0_cv >= umbral).astype(int)
    
    # Calculamos el F1 para la Clase 0
    f1 = f1_score(y_train_binario, predicciones_temporales, average='binary')
    f1_scores_cv.append(f1)

# 5. Obtenemos el umbral ganador
mejor_indice = np.argmax(f1_scores_cv)
mejor_umbral = umbrales[mejor_indice]
mejor_f1 = f1_scores_cv[mejor_indice]

print(f"\n¡Listo! El umbral óptimo encontrado mediante CV es: {mejor_umbral:.2f}")
print(f"F1-Score esperado para la Clase 0 con este umbral: {mejor_f1:.4f}")

# (Opcional) Dibujar la gráfica
plt.figure(figsize=(8, 4))
plt.plot(umbrales, f1_scores_cv, color='purple')
plt.axvline(x=mejor_umbral, color='red', linestyle='--', label=f'Umbral óptimo: {mejor_umbral:.2f}')
plt.title('Optimización de Umbral para Clase 0 (Cross-Validation)')
plt.xlabel('Umbral')
plt.ylabel('F1-Score')
plt.legend()
plt.show()

print("\n¡Modelo entrenado y listo para predecir!")


Los mejores parámetros encontrados fueron:
{'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0}
Calculando probabilidades out-of-fold (esto puede tardar un poco)...


TypeError: got an unexpected keyword argument 'cat_features'